In [ ]:
import pandas as pd
pd.set_option("display.max_columns", None)
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style = "darkgrid")
import numpy as np

In [ ]:
evaluation_df = pd.read_parquet("../data/claims_evaluation.parquet")

## HAS FEATURE ENGINEERING IMPROVED MODELS?
For linear regression, the added features in each version led to a decrease in model performance. Comparing the training and testing metrics for these models, the performance gets worse as they start overfitting to the training data. For the random forest models, each new feature slightly improves model performance per dataset version. For the gradient boosting models, the first added feature noticeably improves model performance. However, once the third feature is added to dataset 3, model performance decreases. This decrease is minor and is better than the original dataset. Overall, feature engineering presents interesting results as each model reacts very differently to the new features. 

In [ ]:
best_model_per_version = evaluation_df.loc[[0, 2, 4, 5, 7, 9, 10, 12, 14]]
metrics = ["rmse", "mae", "r2"]

fig, axes = plt.subplots(1, 3, figsize = (10, 4), constrained_layout = True)
axes = axes.flatten() 
plt.suptitle("Best Model Claim Metrics per Dataset Version", fontsize = 14, fontweight = "semibold")

for ax, column in zip(axes, metrics):
    sns.barplot(data = best_model_per_version, x = column, y = "model", hue = "dataset_version", ax = ax)
    diff = (max(best_model_per_version[column]) - min(best_model_per_version[column])) * 0.1
    
    ax.set_xlim(min(best_model_per_version[column]) - diff, max(best_model_per_version[column]) + diff)

axes[2].legend(title = "Dataset Version", bbox_to_anchor = (1.65, 1))

for ax in axes[:-1]:
    ax.get_legend().remove()

for ax in axes[1:]:
    ax.yaxis.set_visible(False)

fig.savefig("../figures/claims_model_metrics_per_dataset.png", dpi = 300)
plt.show()

## BEST MODEL COMPARISON
From the bar chart, the random forest model is the overall best with the lowest RMSE and MAE, and the highest $R^{2}$. The gradient boosting model is the second best and close behind the random forest model. The linear regression model is by far the worst, with RMSE and MAE values over 1000 more than the random forest.

Looking at the predicted claim amounts, the linear regression model presents two modes representing the bimodal data seen in the data analysis. The predicted larger claim amounts seem to be distributed randomly, especially at 25000 to 50000 and 70000 plus, although predictions around 50000 to 65000 look mostly accurate. The actual smaller claims are distributed from 1000 to 10000, but the predictions made by the model range from 1000 to 5000. 

This is similar to the random forest model, which distributes predictions across two modes. Here, the predicted smaller claims are far less spread, meaning they are less accurate than the linear regression model. The larger predicted claims are also less distributed, mainly ranging from 57500 to 65000. The mean of the actual larger claims is around 63000, meaning the model is mainly predicting the mean. 

The gradient boosting model predictions are similar to both, being more distributed than the random forest predictions but less than the linear regression. What is interesting about these models is that even though linear regression has by far the worst metrics, looking at these graphs, it does not seem to perform that poorly.

When analysing a snapshot of the predictions for each model against the real claim amounts, we see the same pattern where random forest and gradient boosting are distributed mainly around the 60000 claim amount. Although the linear regression model does not appear to be too poor in the actual claims vs predicted claims graph, here it is by far the worst, with scattered guesses that stray far from the true claim amount. All predictions for the lower claim amounts look the same for each model, regardless of the actual small claim amount. 

In [ ]:
best_models = evaluation_df.loc[[0, 12, 9]].reset_index(drop = True)
y_test = best_models["y_test"][0]

In [ ]:
metrics = ["rmse", "mae", "r2"]

fig, axes = plt.subplots(1, 3, figsize = (10, 3), constrained_layout = True)
axes = axes.flatten() 
plt.suptitle("Best Model Claim Metrics", fontsize = 14, fontweight = "semibold")

for ax, column in zip(axes, metrics):
    sns.barplot(data = best_models, x = column, y = "model", hue = "model", ax = ax)
    diff = (max(best_models[column]) - min(best_models[column])) * 0.1
    
    ax.set_xlim(min(best_models[column]) - diff, max(best_models[column]) + diff)
    
for ax in axes[1:]:
    ax.yaxis.set_visible(False)

fig.savefig("../figures/fraud_metrics_by_model.png", dpi = 300)
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize = (12, 4), constrained_layout = True)
axes = axes.flatten() 
plt.suptitle("Actual Claim vs Predicted Claim", fontsize = 14, fontweight = "semibold")

for ax, (index, row) in zip(axes, best_models.iterrows()):
    sns.regplot(x = row["y_test"], y = row["y_test_predictions"], ax = ax, line_kws = {"color": "red"}, scatter_kws = {"s": 35, "alpha": 0.75})
    ax.set_title(row["model"])
    ax.set_xlabel("Actual Claim Amount")
    ax.set_yticks(np.arange(0, 100001, 20000))

axes[0].set_ylabel("Predicted Claim Amount")

fig.savefig("../figures/fraud_actual_vs_predicted.png", dpi = 300)
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 1, figsize = (10, 3), constrained_layout = True)
plt.suptitle("Model Prediction Snapshot", fontsize = 14, fontweight = "semibold")

axes.scatter(x = np.arange(0, 50), y = y_test[50:100], label = "Actual Claim Amount")

for index, row in best_models.iterrows():
    sns.scatterplot(x = np.arange(0, 50), y = row["y_test_predictions"][50:100], label = row["model"], ax = axes, alpha = 0.75)

axes.legend(bbox_to_anchor = (1, 1))
axes.set_ylabel("Claim Amount")

fig.savefig("../figures/fraud_prediction_snapshot.png", dpi = 300)
plt.show()

## THE BEST MODEL FEATURES
In the best model, the graph suggests that the majority of the prediction is based on a single feature, 'collision_type_unknown', and the rest is handled across four other features, three of which are 'incident_type' values. What the model is most likely doing is first checking whether the total claim amount is part of the high or low claims, and then slightly adjusting afterwards with other features. This is the most probable case, as the predictions the model makes are highly concentrated near each claim. 

In [ ]:
best_model_features = pd.read_parquet("../data/best_regression_model_features.parquet")
best_model_features = best_model_features.rename(columns = {"Column": "Feature"})

fig, axes = plt.subplots(1, 1, figsize = (12, 4), constrained_layout = True)
plt.suptitle("Random Forest - Feature Importance", fontsize = 14, fontweight = "semibold")

best_model_features.sort_values("Feature Importance", ascending = False).head(10).plot(kind = "barh", y = "Feature Importance", 
                                                                                       x = "Feature", ax = axes, legend = False)

fig.savefig("../figures/fraud_best_model_feature_importance.png", dpi = 300)
plt.show()

##  CONCLUSION & LIMITATIONS
Overall, I would classify these models as underperforming. While they demonstrate an ability to differentiate between high and low claims, the predictions lack meaningful accuracy, as they mostly cluster around the mean. That being said, these models and evaluations show the work is promising, but the methodology needs to change. Instead of a singular model which predicts all total claim amounts, multiple models should be involved. One model should be made to classify claims as high or low, which the current models have already established is possible. Once the claim is classified, separate regression models can be used to predict the expected claim amount. This would enable the new models to focus only on the high or low ranges, and that would mean the final models should predict the actual claim amount instead of resorting to the mean. 

There are, however, limitations to this idea and the current models. First, the dataset is small, meaning that if the data is separated and then split into training and testing sets, the models may learn nothing and interpret mostly noise. Secondly, the distribution of high and low claims is imbalanced, with far more high claims than low ones. This means that there is not enough training data to capture details around low claims. It could be possible to remove the low claims entirely, but that does not meet to inital expectatyions, and defeats the purpose of this project. 